# 📦 Notebook 01 — Data Collection

**Mục tiêu:** Thu thập và lưu trữ toàn bộ dữ liệu thô cần thiết cho đề tài:

| Dữ liệu | Nguồn | Dùng cho |
|---|---|---|
| Giá điều chỉnh OHLCV (ngày) | vnstock / VCI | Lợi suất hàng ngày/tháng, backtest |
| Vốn hóa thị trường (market cap) | vnstock / VCI | Phân loại Small/Big (SMB factor) |
| Vốn chủ sở hữu (equity/book value) | vnstock / VCI | Tính P/B → phân loại HML factor |

> ⚠️ **Lưu ý Point-in-Time:** Book value cuối năm N sẽ được dùng để rebalance nhân tố vào **tháng 7 năm N+1** (chuẩn Fama-French), đảm bảo thông tin đã được công bố.

**Universe:** VN100 (100 mã cổ phiếu lớn nhất HOSE)

**Khoảng thời gian:** 2015-01-01 → nay (~10 năm)

## 0. Setup & Imports

In [ ]:
import os
import time
import warnings
from pathlib import Path
from datetime import datetime, date

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.0f}'.format)

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT_DIR  = Path('..').resolve()
DATA_DIR  = ROOT_DIR / 'data'
RAW_DIR   = DATA_DIR / 'raw'
CLEAN_DIR = DATA_DIR / 'clean'

for d in [RAW_DIR, CLEAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Root  : {ROOT_DIR}")
print(f"Raw   : {RAW_DIR}")
print(f"Clean : {CLEAN_DIR}")

In [ ]:
# ── vnstock new API ──────────────────────────────────────────────────────────
from vnstock import Listing
from vnstock.api.quote   import Quote
from vnstock.api.financial import Finance
from vnstock.api.company import Company

# ── Config ──────────────────────────────────────────────────────────────────
SOURCE      = 'VCI'           # Data source
START_DATE  = '2015-01-01'    # Lấy từ 2015 (~10 năm)
END_DATE    = date.today().strftime('%Y-%m-%d')
INTERVAL    = '1D'            # Dữ liệu ngày
SLEEP_SEC   = 0.5             # Delay giữa các request để tránh bị rate-limit

print(f"Thu thập dữ liệu từ {START_DATE} đến {END_DATE}")

## 1. Lấy danh sách mã VN100

In [ ]:
listing = Listing()
vn100_series = listing.symbols_by_group('VN100')
SYMBOLS = sorted(vn100_series.tolist())

print(f"✅ VN100 universe: {len(SYMBOLS)} mã")
print(SYMBOLS)

In [ ]:
# Lưu danh sách mã vào file để reference
pd.Series(SYMBOLS, name='symbol').to_csv(RAW_DIR / 'vn100_symbols.csv', index=False)
print("Lưu danh sách mã VN100 thành công!")

## 2. Thu thập giá lịch sử OHLCV (Giá điều chỉnh)

> vnstock VCI trả về giá đã điều chỉnh (adjusted) theo mặc định — không cần xử lý thêm.
> Xác nhận lại bằng cách so sánh chéo với CafeF cho vài mã sau khi hoàn thành.

In [ ]:
def fetch_price_history(symbol: str, start: str, end: str,
                         source: str = 'VCI', interval: str = '1D',
                         max_retries: int = 3) -> pd.DataFrame | None:
    """Lấy giá lịch sử 1 mã. Trả về None nếu lỗi sau max_retries lần."""
    for attempt in range(max_retries):
        try:
            q  = Quote(symbol=symbol, source=source)
            df = q.history(start=start, end=end, interval=interval)
            if df is not None and not df.empty:
                df.insert(0, 'symbol', symbol)
                df.rename(columns={'time': 'date'}, inplace=True)
                df['date'] = pd.to_datetime(df['date'])
                df.set_index('date', inplace=True)
                return df
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(SLEEP_SEC * 2)
            else:
                print(f"  ⚠️ FAIL {symbol}: {e}")
    return None

In [ ]:
# ── Test với 1 mã trước ────────────────────────────────────────────────────
test_df = fetch_price_history('VNM', START_DATE, END_DATE)
print(test_df.head())
print(f"Shape: {test_df.shape}")
print(f"Date range: {test_df.index.min()} → {test_df.index.max()}")

In [ ]:
# ── Pull toàn bộ VN100 ────────────────────────────────────────────────────
all_prices = []
failed_symbols = []

for sym in tqdm(SYMBOLS, desc='Pulling OHLCV'):
    df = fetch_price_history(sym, START_DATE, END_DATE, source=SOURCE, interval=INTERVAL)
    if df is not None:
        all_prices.append(df)
    else:
        failed_symbols.append(sym)
    time.sleep(SLEEP_SEC)

print(f"\n✅ Thành công: {len(all_prices)}/{len(SYMBOLS)} mã")
if failed_symbols:
    print(f"⚠️ Thất bại   : {failed_symbols}")

In [ ]:
# ── Gộp và lưu ────────────────────────────────────────────────────────────
price_df = pd.concat(all_prices)
price_df = price_df.reset_index()
price_df = price_df.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f"Tổng số hàng  : {len(price_df):,}")
print(f"Số mã         : {price_df['symbol'].nunique()}")
print(f"Khoảng thời gian: {price_df['date'].min()} → {price_df['date'].max()}")
price_df.head(10)

In [ ]:
# Lưu raw
price_df.to_parquet(RAW_DIR / 'ohlcv_raw.parquet', index=False)
print(f"✅ Đã lưu: {RAW_DIR / 'ohlcv_raw.parquet'}")

## 3. Thu thập Market Cap (Vốn hóa thị trường)

Dùng `Company.trading_stats()` — trả về `market_cap` và số lượng cổ phiếu lưu hành.

> **Lưu ý:** API này chỉ trả về snapshot **tại thời điểm hiện tại**. Để có lịch sử market cap theo tháng, chúng ta sẽ tính: `market_cap_monthly = close_price × shares_outstanding` từ dữ liệu giá lịch sử và số cổ phiếu lưu hành hiện tại (chấp nhận gần đúng).

In [ ]:
def fetch_trading_stats(symbol: str, source: str = 'VCI',
                         max_retries: int = 3) -> dict | None:
    """Lấy trading stats (market cap, shares outstanding) hiện tại."""
    for attempt in range(max_retries):
        try:
            c  = Company(symbol=symbol, source=source)
            df = c.trading_stats()
            if df is not None and not df.empty:
                row = df.iloc[0]
                return {
                    'symbol'             : symbol,
                    'market_cap'         : row.get('market_cap', None),
                    'shares_outstanding' : row.get('number_of_shares_mkt_cap', None),
                    'current_price'      : row.get('current_price', None),
                    'free_float'         : row.get('free_float_percentage', None),
                }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(SLEEP_SEC * 2)
            else:
                print(f"  ⚠️ FAIL {symbol}: {e}")
    return None

In [ ]:
mktcap_records = []
for sym in tqdm(SYMBOLS, desc='Pulling Market Cap'):
    record = fetch_trading_stats(sym, source=SOURCE)
    if record:
        mktcap_records.append(record)
    time.sleep(SLEEP_SEC)

mktcap_df = pd.DataFrame(mktcap_records)
print(f"✅ Market cap thu thập được: {len(mktcap_df)} mã")
mktcap_df.head(10)

In [ ]:
mktcap_df.to_csv(RAW_DIR / 'market_cap_snapshot.csv', index=False)
print(f"✅ Đã lưu: {RAW_DIR / 'market_cap_snapshot.csv'}")

## 4. Thu thập Book Value (Vốn chủ sở hữu từ Bảng cân đối kế toán)

Lấy **Vốn chủ sở hữu (Owner's Equity)** từ bảng cân đối kế toán hàng năm.

- Dòng cần lấy: `Vốn chủ sở hữu` (item_id tương ứng)
- Theo chuẩn Fama-French: dùng book value **cuối năm tài chính T** để tính nhân tố HML cho kỳ **tháng 7 năm T+1 đến tháng 6 năm T+2**.

In [ ]:
def fetch_book_value(symbol: str, source: str = 'VCI',
                      max_retries: int = 3) -> pd.DataFrame | None:
    """
    Lấy lịch sử Vốn chủ sở hữu (Owner's Equity) theo năm.
    Trả về DataFrame có cột: symbol, year, book_equity (đơn vị: VND)
    """
    for attempt in range(max_retries):
        try:
            f  = Finance(symbol=symbol, source=source, period='year')
            bs = f.balance_sheet(lang='en')
            if bs is None or bs.empty:
                return None

            # Tìm dòng Owner's Equity (Vốn chủ sở hữu)
            mask = bs['item'].str.contains('Vốn chủ sở hữu', na=False)
            equity_rows = bs[mask]

            if equity_rows.empty:
                # Fallback: tổng nguồn vốn - nợ phải trả (nếu không tìm thấy trực tiếp)
                mask2 = bs['item'].str.contains('Tổng cộng nguồn vốn', na=False)
                equity_rows = bs[mask2]

            if equity_rows.empty:
                return None

            # Lấy dòng đầu tiên, các cột là năm
            row = equity_rows.iloc[0]
            year_cols = [c for c in bs.columns if str(c).isdigit()]

            records = []
            for year in year_cols:
                val = row.get(year, None)
                try:
                    val = float(val)
                except:
                    val = None
                records.append({'symbol': symbol, 'year': int(year), 'book_equity': val})

            return pd.DataFrame(records)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(SLEEP_SEC * 2)
            else:
                print(f"  ⚠️ FAIL {symbol}: {e}")
    return None

In [ ]:
# ── Test với 1 mã ─────────────────────────────────────────────────────────
test_bv = fetch_book_value('VNM', source=SOURCE)
print(test_bv)

In [ ]:
# ── Pull toàn bộ VN100 ────────────────────────────────────────────────────
all_bv  = []
fail_bv = []

for sym in tqdm(SYMBOLS, desc='Pulling Book Value'):
    df = fetch_book_value(sym, source=SOURCE)
    if df is not None:
        all_bv.append(df)
    else:
        fail_bv.append(sym)
    time.sleep(SLEEP_SEC)

book_value_df = pd.concat(all_bv).reset_index(drop=True)
book_value_df = book_value_df.sort_values(['symbol', 'year']).reset_index(drop=True)

print(f"\n✅ Book value thu thập được: {book_value_df['symbol'].nunique()} mã")
if fail_bv:
    print(f"⚠️ Thất bại: {fail_bv}")
book_value_df.head(10)

In [ ]:
book_value_df.to_csv(RAW_DIR / 'book_value_annual.csv', index=False)
print(f"✅ Đã lưu: {RAW_DIR / 'book_value_annual.csv'}")

## 5. Tạo Dataset Tổng Hợp Theo Tháng (Clean Dataset)

Gộp dữ liệu thành 1 bảng master theo tần suất **tháng** (resample từ dữ liệu ngày).

Mỗi hàng = 1 cổ phiếu × 1 tháng, gồm:
- `close_eom`: Giá đóng cửa cuối tháng (End-of-Month)
- `monthly_return`: Lợi suất tháng (log return)
- `volume_avg`: Khối lượng giao dịch trung bình trong tháng
- `market_cap_eom`: Vốn hóa cuối tháng = `close_eom × shares_outstanding`
- `book_equity`: Vốn CSH cuối năm (theo chuẩn point-in-time Fama-French)

In [ ]:
# ── Resample giá về tần suất tháng ────────────────────────────────────────
price_df['date'] = pd.to_datetime(price_df['date'])
price_df = price_df.sort_values(['symbol', 'date'])

def resample_monthly(group: pd.DataFrame) -> pd.DataFrame:
    group = group.set_index('date').sort_index()
    monthly = group.resample('ME').agg(
        close_eom     = ('close', 'last'),
        open_bom      = ('open', 'first'),
        high_month    = ('high', 'max'),
        low_month     = ('low', 'min'),
        volume_avg    = ('volume', 'mean'),
    )
    # Lợi suất tháng: log return dùng giá đóng cửa cuối tháng
    monthly['monthly_return'] = np.log(monthly['close_eom'] / monthly['close_eom'].shift(1))
    monthly.index.name = 'date'
    monthly = monthly.reset_index()
    monthly.insert(0, 'symbol', group['symbol'].iloc[0])
    return monthly

monthly_price = (
    price_df.groupby('symbol', group_keys=False)
            .apply(resample_monthly)
            .reset_index(drop=True)
)

print(f"Monthly price shape: {monthly_price.shape}")
monthly_price.head(10)

In [ ]:
# ── Ghép shares_outstanding từ market cap snapshot ─────────────────────────
shares_map = mktcap_df.set_index('symbol')['shares_outstanding'].to_dict()

monthly_price['shares_outstanding'] = monthly_price['symbol'].map(shares_map)
monthly_price['market_cap_eom']     = monthly_price['close_eom'] * monthly_price['shares_outstanding']

print("Đã tính market_cap_eom = close_eom × shares_outstanding")
monthly_price[['symbol', 'date', 'close_eom', 'shares_outstanding', 'market_cap_eom']].head(5)

In [ ]:
# ── Ghép book value theo chuẩn Point-in-Time Fama-French ──────────────────
# Quy tắc: Book value của năm T → dùng cho kỳ tháng 7/T+1 đến tháng 6/T+2
# Cách implement: book_value năm T → gán cho 'ff_year' = T+1
book_value_df['ff_year'] = book_value_df['year'] + 1  # Point-in-time shift

monthly_price['year']    = monthly_price['date'].dt.year
monthly_price['month']   = monthly_price['date'].dt.month

# Xác định 'ff_year' cho từng tháng:
# Tháng 7-12 của năm Y → ff_year = Y (dùng book value năm Y-1+1 = Y)
# Tháng 1-6  của năm Y → ff_year = Y (vẫn dùng book value năm Y-1)
monthly_price['ff_year'] = monthly_price.apply(
    lambda r: r['year'] if r['month'] >= 7 else r['year'],
    axis=1
)
# Tinh chỉnh: tháng 1-6/Y → ff_year = Y (book từ năm Y-1 shift +1 = Y ✓)
# tháng 7-12/Y → ff_year = Y (book từ năm Y-1 shift +1 = Y ✓ — CORRECT)

bv_lookup = book_value_df[['symbol', 'ff_year', 'book_equity']].copy()

monthly_price = monthly_price.merge(
    bv_lookup,
    on=['symbol', 'ff_year'],
    how='left'
)

# Tính Book-to-Market ratio
monthly_price['book_to_market'] = monthly_price['book_equity'] / monthly_price['market_cap_eom']

print(f"Master dataset shape: {monthly_price.shape}")
monthly_price.head()

In [ ]:
# ── Kiểm tra chất lượng dữ liệu ──────────────────────────────────────────
print("=== Kiểm tra missing values ===")
print(monthly_price.isnull().sum())

print(f"\n=== Thống kê tổng quan ===")
print(monthly_price[['close_eom', 'monthly_return', 'market_cap_eom', 'book_equity', 'book_to_market']].describe())

In [ ]:
# ── Lọc và làm sạch ──────────────────────────────────────────────────────
# Loại bỏ tháng đầu tiên của mỗi mã (lợi suất NaN)
monthly_clean = monthly_price.dropna(subset=['monthly_return', 'close_eom']).copy()

# Loại bỏ outlier lợi suất cực đoan (>100% hoặc <-70% trong 1 tháng — có thể lỗi dữ liệu)
ret_mask = (monthly_clean['monthly_return'] > -1.5) & (monthly_clean['monthly_return'] < 1.5)
n_outlier = (~ret_mask).sum()
print(f"Loại bỏ {n_outlier} hàng có lợi suất ngoại lệ")
monthly_clean = monthly_clean[ret_mask].copy()

print(f"Clean dataset shape: {monthly_clean.shape}")

In [ ]:
# ── Sắp xếp cột và lưu ──────────────────────────────────────────────────
final_cols = [
    'symbol', 'date', 'year', 'month', 'ff_year',
    'open_bom', 'close_eom', 'high_month', 'low_month', 'volume_avg',
    'monthly_return',
    'shares_outstanding', 'market_cap_eom',
    'book_equity', 'book_to_market',
]
monthly_clean = monthly_clean[final_cols].sort_values(['symbol', 'date']).reset_index(drop=True)

# Lưu
monthly_clean.to_parquet(CLEAN_DIR / 'monthly_panel.parquet', index=False)
monthly_clean.to_csv(CLEAN_DIR / 'monthly_panel.csv', index=False)

print(f"✅ Đã lưu Clean Dataset:")
print(f"   {CLEAN_DIR / 'monthly_panel.parquet'}")
print(f"   {CLEAN_DIR / 'monthly_panel.csv'}")
print(f"\nShape cuối: {monthly_clean.shape}")
monthly_clean.tail(10)

## 6. Kiểm Tra Chéo (Cross-validation)

So sánh nhanh với dữ liệu CafeF để phát hiện sai lệch ở 2-3 mã tiêu biểu.

In [ ]:
# Kiểm tra mã VNM: in lợi suất năm 2023 để đối chiếu thủ công với CafeF
check = monthly_clean[
    (monthly_clean['symbol'] == 'VNM') &
    (monthly_clean['date'].dt.year == 2023)
][['symbol', 'date', 'close_eom', 'monthly_return', 'market_cap_eom']]

check['monthly_return_pct'] = (check['monthly_return'] * 100).round(2)
print("Lợi suất VNM năm 2023 (dùng để đối chiếu CafeF):")
print(check.to_string(index=False))

## 7. Tóm tắt Files Đầu Ra

| File | Mô tả |
|---|---|
| `data/raw/vn100_symbols.csv` | Danh sách 100 mã VN100 |
| `data/raw/ohlcv_raw.parquet` | Giá OHLCV ngày, toàn bộ VN100, ~10 năm |
| `data/raw/market_cap_snapshot.csv` | Snapshot market cap + shares outstanding hiện tại |
| `data/raw/book_value_annual.csv` | Vốn CSH theo năm từ BCTC |
| `data/clean/monthly_panel.parquet` | **Master dataset** — panel data tháng, đã sạch |
| `data/clean/monthly_panel.csv` | Bản CSV của master dataset |

> ✅ **Dataset sẵn sàng cho Notebook 02** — Factor Construction (SMB, HML, MKT)